# Qwen3 Q/K capture smoke test (Colab)

This notebook is a small Colab smoke test for `src/dataset_generation/qk_hook_attention/capture_qk_qwen3.py` on `Qwen/Qwen3-8B`. It captures raw Q/K projection tensors for layers `[0, 20, 35]`, saves them in `qk_cache_test`, then loads a few tensors back and prints summary statistics.

Run the cells top to bottom. The model is large, so use a GPU runtime with enough memory (A100/L4 recommended). The default attention implementation is `sdpa` because it works on standard Colab images without compiling FlashAttention; change `ATTN_IMPLEMENTATION` to `flash_attention_2` if your runtime has `flash-attn` installed.


## 1. Install/check dependencies

This cell checks whether common dependencies are importable and installs only missing packages. PyTorch is normally preinstalled on Colab; if it is missing, install it manually from the official PyTorch instructions for your CUDA runtime.


In [1]:
import importlib.util
import subprocess
import sys

packages = {
    'torch': 'torch',
    'transformers': 'transformers',
    'accelerate': 'accelerate',
    'huggingface_hub': 'huggingface_hub',
    'sentencepiece': 'sentencepiece',
    'safetensors': 'safetensors',
}

missing = [pip_name for module_name, pip_name in packages.items() if importlib.util.find_spec(module_name) is None]
print('Missing packages:', missing or 'none')

# Colab usually already has torch. Avoid silently installing an incompatible CUDA build.
if 'torch' in missing:
    raise RuntimeError(
        'torch is not installed. Install a CUDA-compatible PyTorch build first, then rerun this cell.'
    )

to_install = [pkg for pkg in missing if pkg != 'torch']
if to_install:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', *to_install])

import torch
import transformers
import accelerate
import huggingface_hub

print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('accelerate:', accelerate.__version__)
print('huggingface_hub:', huggingface_hub.__version__)
print('CUDA available:', torch.cuda.is_available())


Missing packages: none
torch: 2.11.0+cu128
transformers: 5.0.0
accelerate: 1.13.0
huggingface_hub: 1.16.1
CUDA available: True


## 2. Print GPU name and memory


In [2]:
import subprocess

try:
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
        check=True,
        text=True,
        capture_output=True,
    )
    print(result.stdout.strip())
except Exception as exc:
    print('Could not run nvidia-smi:', repr(exc))


NVIDIA A100-SXM4-40GB, 40960 MiB, 40437 MiB


## 3. Mount Drive / choose repository path

Set `REPO_DIR` to the folder that contains this repository. If you uploaded or cloned the repo directly into `/content`, update the path accordingly.


In [3]:
from pathlib import Path
import os
import sys
from google.colab import drive
drive.mount('/content/drive')

# CHANGE THIS if your checkout is somewhere else.
REPO_DIR = Path('/content/drive/MyDrive/Colab Notebooks/compression/dataset-generation-main-v5')

os.chdir(REPO_DIR)
if str(REPO_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_DIR / 'src'))

SCRIPT_PATH = REPO_DIR / 'src' / 'dataset_generation' / 'qk_hook_attention' / 'capture_qk_qwen3.py'

print('Working directory:', Path.cwd())
print('Capture script:', SCRIPT_PATH)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/Colab Notebooks/compression/dataset-generation-main-v5
Capture script: /content/drive/MyDrive/Colab Notebooks/compression/dataset-generation-main-v5/src/dataset_generation/qk_hook_attention/capture_qk_qwen3.py


## 4. Hugging Face login (if needed)

This does not hard-code tokens. If your runtime is already authenticated or the model is accessible without auth, the cell prints the current status and continues. Otherwise it prompts for a Hugging Face token.


In [4]:
"""
from huggingface_hub import HfFolder, notebook_login

token = HfFolder.get_token()
if token:
    print('Hugging Face token is already available in this runtime.')
else:
    print('No Hugging Face token found. Run notebook_login() and paste a token if needed.')
    notebook_login()
"""

"\nfrom huggingface_hub import HfFolder, notebook_login\n\ntoken = HfFolder.get_token()\nif token:\n    print('Hugging Face token is already available in this runtime.')\nelse:\n    print('No Hugging Face token found. Run notebook_login() and paste a token if needed.')\n    notebook_login()\n"

## 5. Configure a short prompt and output directory

The prompt below is intentionally short (roughly around 128 tokens after tokenization, depending on tokenizer/template settings). The capture command writes to the `qk_cache_test` subdirectory.


In [5]:
from pathlib import Path
import shutil
from transformers import AutoTokenizer

MODEL_NAME = 'Qwen/Qwen3-8B'
TARGET_LAYERS = [0, 20, 35]
SAVE_DTYPE = 'bfloat16'
ATTN_IMPLEMENTATION = 'sdpa'  # Change to 'flash_attention_2' if flash-attn is installed in your runtime.
OUT_DIR = Path('qk_cache_test')
PROMPT_FILE = Path('qk_smoke_prompt.txt')

SHORT_PROMPT = '''
You are helping with a quick attention-capture smoke test. Read this short paragraph carefully, keep the details in order, and answer only after thinking about the sequence. The red notebook is on the desk, the brass key is under the cup, and the final password is glacier. A courier named Mira arrives at noon, checks the window latch, copies the password into a blue envelope, and leaves the envelope beside the lamp. Summarize the important remembered facts in one concise sentence, preserving the objects, locations, name, time, and password.
'''.strip()

PROMPT_FILE.write_text(SHORT_PROMPT, encoding='utf-8')

# Optional cleanup so this smoke test starts from a fresh output folder.
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
prompt_ids = tokenizer(SHORT_PROMPT, add_special_tokens=False).input_ids
print('Prompt file:', PROMPT_FILE)
print('Output directory:', OUT_DIR)
print('Approx prompt token count without chat template:', len(prompt_ids))
print('Target layers:', TARGET_LAYERS)
print('Expected Qwen3-8B shapes after tokenization:')
print('  q_raw: [1, T, 4096]')
print('  k_raw: [1, T, 1024]')
print('  where T is the tokenized sequence length printed in metadata.json after capture.')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Prompt file: qk_smoke_prompt.txt
Output directory: qk_cache_test
Approx prompt token count without chat template: 111
Target layers: [0, 20, 35]
Expected Qwen3-8B shapes after tokenization:
  q_raw: [1, T, 4096]
  k_raw: [1, T, 1024]
  where T is the tokenized sequence length printed in metadata.json after capture.


## 6. Run `capture_qk_qwen3.py`

This runs one forward pass with hooks on layers `0,20,35` and saves raw Q/K projection tensors as bfloat16.


In [6]:
import subprocess
import sys

cmd = [
    sys.executable,
    str(SCRIPT_PATH),
    '--model-name', MODEL_NAME,
    '--prompt-file', str(PROMPT_FILE),
    '--out-dir', str(OUT_DIR),
    '--layers', ','.join(map(str, TARGET_LAYERS)),
    '--attn-implementation', ATTN_IMPLEMENTATION,
    '--model-dtype', 'bfloat16',
    '--save-dtype', SAVE_DTYPE,
]

print('Running command:')
print(' '.join(cmd))
try:
    result = subprocess.run(
        cmd,
        check=True,
        capture_output=True,
        text=True,
    )
    print(result.stdout)

except subprocess.CalledProcessError as e:
    print("Command failed!")
    print("Return code:", e.returncode)

    print("\n===== STDOUT =====")
    print(e.stdout)

    print("\n===== STDERR =====")
    print(e.stderr)

    raise


Running command:
/usr/bin/python3 /content/drive/MyDrive/Colab Notebooks/compression/dataset-generation-main-v5/src/dataset_generation/qk_hook_attention/capture_qk_qwen3.py --model-name Qwen/Qwen3-8B --prompt-file qk_smoke_prompt.txt --out-dir qk_cache_test --layers 0,20,35 --attn-implementation sdpa --model-dtype bfloat16 --save-dtype bfloat16
[load] Qwen/Qwen3-8B with attn_implementation=sdpa
[hook] saved layer=0 q_raw shape=(1, 111, 4096) dtype=torch.bfloat16 -> qk_cache_test/layer_00_q_raw.pt
[hook] saved layer=0 k_raw shape=(1, 111, 1024) dtype=torch.bfloat16 -> qk_cache_test/layer_00_k_raw.pt
[hook] saved layer=20 q_raw shape=(1, 111, 4096) dtype=torch.bfloat16 -> qk_cache_test/layer_20_q_raw.pt
[hook] saved layer=20 k_raw shape=(1, 111, 1024) dtype=torch.bfloat16 -> qk_cache_test/layer_20_k_raw.pt
[hook] saved layer=35 q_raw shape=(1, 111, 4096) dtype=torch.bfloat16 -> qk_cache_test/layer_35_q_raw.pt
[hook] saved layer=35 k_raw shape=(1, 111, 1024) dtype=torch.bfloat16 -> qk_cac

## 7. List saved files


In [7]:
from pathlib import Path

for path in sorted(OUT_DIR.iterdir()):
    size_mb = path.stat().st_size / (1024 ** 2)
    print(f'{path.name:32s} {size_mb:10.3f} MB')


analysis_spec.json                    0.000 MB
attention_mask.pt                     0.002 MB
input_ids.pt                          0.002 MB
layer_00_k_raw.pt                     0.218 MB
layer_00_q_raw.pt                     0.869 MB
layer_00_qk_norms.pt                  0.003 MB
layer_20_k_raw.pt                     0.218 MB
layer_20_q_raw.pt                     0.869 MB
layer_20_qk_norms.pt                  0.003 MB
layer_35_k_raw.pt                     0.218 MB
layer_35_q_raw.pt                     0.869 MB
layer_35_qk_norms.pt                  0.003 MB
metadata.json                         0.003 MB
model_text.txt                        0.001 MB
position_ids.pt                       0.002 MB
prompt.txt                            0.001 MB
tokens.json                           0.001 MB


## 8. Load `metadata.json` and print important fields


In [8]:
import json

metadata_path = OUT_DIR / 'metadata.json'
metadata = json.loads(metadata_path.read_text(encoding='utf-8'))

important_fields = [
    'model_name',
    'transformers_version',
    'torch_version',
    'attn_implementation_requested',
    'attn_implementation_model_config',
    'model_dtype',
    'save_dtype',
    'target_layers',
    'seq_len',
    'batch_size',
]

for field in important_fields:
    print(f'{field}: {metadata.get(field)}')

config = metadata.get('model_config', {})
print('num_hidden_layers:', config.get('num_hidden_layers'))
print('hidden_size:', config.get('hidden_size'))
print('num_attention_heads:', config.get('num_attention_heads'))
print('num_key_value_heads:', config.get('num_key_value_heads'))
print('head_dim:', config.get('head_dim'))


model_name: Qwen/Qwen3-8B
transformers_version: 5.0.0
torch_version: 2.11.0+cu128
attn_implementation_requested: sdpa
attn_implementation_model_config: sdpa
model_dtype: torch.bfloat16
save_dtype: torch.bfloat16
target_layers: [0, 20, 35]
seq_len: 111
batch_size: 1
num_hidden_layers: 36
hidden_size: 4096
num_attention_heads: 32
num_key_value_heads: 8
head_dim: 128


## 9. Estimate expected tensor shapes for Qwen3-8B

For Qwen3-8B, the expected raw projection widths are:

- `q_raw`: `[1, T, 4096]`
- `k_raw`: `[1, T, 1024]`

where `T` is the tokenized sequence length from `metadata.json`. This cell computes those expected shapes from the saved metadata.


In [9]:
T = int(metadata['seq_len'])
expected = {
    'q_raw': (1, T, 4096),
    'k_raw': (1, T, 1024),
}
print('T from metadata:', T)
print('Expected q_raw shape:', expected['q_raw'])
print('Expected k_raw shape:', expected['k_raw'])


T from metadata: 111
Expected q_raw shape: (1, 111, 4096)
Expected k_raw shape: (1, 111, 1024)


## 10. Load selected tensors and print summaries

This loads `layer_00_q_raw.pt`, `layer_00_k_raw.pt`, `layer_35_q_raw.pt`, and `layer_35_k_raw.pt`, then prints shape, dtype, device, min, max, and norm.


In [10]:
import torch

def summarize_tensor(path: Path):
    tensor = torch.load(path, map_location='cpu')
    stats_tensor = tensor.float()
    print(f'{path.name}')
    print('  shape:', tuple(tensor.shape))
    print('  dtype:', tensor.dtype)
    print('  device:', tensor.device)
    print('  min:', float(stats_tensor.min()))
    print('  max:', float(stats_tensor.max()))
    print('  norm:', float(stats_tensor.norm()))
    return tensor

loaded = {}
for name in [
    'layer_00_q_raw.pt',
    'layer_00_k_raw.pt',
    'layer_35_q_raw.pt',
    'layer_35_k_raw.pt',
]:
    loaded[name] = summarize_tensor(OUT_DIR / name)

assert tuple(loaded['layer_00_q_raw.pt'].shape) == expected['q_raw']
assert tuple(loaded['layer_00_k_raw.pt'].shape) == expected['k_raw']
assert tuple(loaded['layer_35_q_raw.pt'].shape) == expected['q_raw']
assert tuple(loaded['layer_35_k_raw.pt'].shape) == expected['k_raw']
print('Shape checks passed.')


layer_00_q_raw.pt
  shape: (1, 111, 4096)
  dtype: torch.bfloat16
  device: cpu
  min: -0.5390625
  max: 0.94921875
  norm: 35.9488525390625
layer_00_k_raw.pt
  shape: (1, 111, 1024)
  dtype: torch.bfloat16
  device: cpu
  min: -0.8203125
  max: 0.58203125
  norm: 23.852027893066406
layer_35_q_raw.pt
  shape: (1, 111, 4096)
  dtype: torch.bfloat16
  device: cpu
  min: -90.0
  max: 146.0
  norm: 4626.953125
layer_35_k_raw.pt
  shape: (1, 111, 1024)
  dtype: torch.bfloat16
  device: cpu
  min: -86.0
  max: 76.5
  norm: 2010.8212890625
Shape checks passed.


---

# Qwen3 Q/K analyzer smoke test

The remaining cells continue from the capture smoke test above and exercise `src/dataset_generation/qk_hook_attention/analyze_qk_qwen3.py` against the generated `qk_cache_test` directory. They are intentionally small so you can rerun individual cells while changing `LAYER_IDX`, `HEAD_IDX`, query positions, or named spans.


## 11. Locate and inspect the cache directory

This cell uses `CACHE_DIR = Path("qk_cache_test")`, checks for `metadata.json`, lists saved artifacts, and prints Qwen/Qwen3-8B shape expectations for raw Q/K tensors.


In [ ]:
from pathlib import Path
import json

CACHE_DIR = Path('qk_cache_test')
metadata_path = CACHE_DIR / 'metadata.json'
assert metadata_path.exists(), f'Missing {metadata_path}; run the capture cells first.'

print('Cache directory:', CACHE_DIR.resolve())
print('\nFiles:')
for path in sorted(CACHE_DIR.iterdir()):
    size_mb = path.stat().st_size / (1024 ** 2)
    print(f'  {path.name:36s} {size_mb:10.3f} MiB')

metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
config = metadata.get('model_config', {})
important = {
    'model_name': metadata.get('model_name'),
    'seq_len': metadata.get('seq_len'),
    'target_layers': metadata.get('target_layers'),
    'num_hidden_layers': config.get('num_hidden_layers'),
    'num_attention_heads': config.get('num_attention_heads'),
    'num_key_value_heads': config.get('num_key_value_heads'),
    'head_dim': config.get('head_dim') or config.get('hidden_size', 0) // max(1, config.get('num_attention_heads', 1)),
    'dtype': metadata.get('save_dtype') or metadata.get('model_dtype'),
}
print('\nMetadata fields:')
for key, value in important.items():
    print(f'  {key}: {value}')

prompt_preview = metadata.get('input_text') or metadata.get('prompt') or metadata.get('prompt_preview')
if prompt_preview:
    preview = str(prompt_preview).replace('\n', ' ')
    print('\nPrompt preview:', preview[:500] + ('...' if len(preview) > 500 else ''))
else:
    print('\nPrompt preview: not stored in metadata')

T = int(metadata['seq_len'])
print('\nExpected raw tensor shapes for Qwen/Qwen3-8B:')
print('  q_raw:', (1, T, 4096))
print('  k_raw:', (1, T, 1024))


## 12. Inspect one raw Q/K layer interactively

Change `LAYER_IDX` and rerun this cell to inspect a different saved layer. The summaries include shape, dtype, device, memory size, scalar statistics, and per-position last-dimension norms.


In [ ]:
import torch

LAYER_IDX = 0  # Change this to any layer captured in CACHE_DIR, e.g. 20 or 35.
q_path = CACHE_DIR / f'layer_{LAYER_IDX:02d}_q_raw.pt'
k_path = CACHE_DIR / f'layer_{LAYER_IDX:02d}_k_raw.pt'
assert q_path.exists(), f'Missing {q_path}'
assert k_path.exists(), f'Missing {k_path}'

def tensor_mib(tensor):
    return tensor.numel() * tensor.element_size() / (1024 ** 2)

def inspect_raw_tensor(name, tensor, positions):
    stats = tensor.float()
    print(f'{name}:')
    print('  shape:', tuple(tensor.shape))
    print('  dtype:', tensor.dtype)
    print('  device:', tensor.device)
    print(f'  memory: {tensor_mib(tensor):.3f} MiB')
    print(f'  mean/std/min/max: {stats.mean().item():.6g} / {stats.std().item():.6g} / {stats.min().item():.6g} / {stats.max().item():.6g}')
    print('  selected position norms:')
    for pos in positions:
        pos = int(max(0, min(tensor.shape[1] - 1, pos)))
        norm = stats[0, pos].norm().item()
        print(f'    pos {pos:5d}: norm={norm:.6g}')

q_raw = torch.load(q_path, map_location='cpu')
k_raw = torch.load(k_path, map_location='cpu')
positions_to_show = sorted({0, T // 4, T // 2, max(0, T - 1)})
inspect_raw_tensor(q_path.name, q_raw, positions_to_show)
inspect_raw_tensor(k_path.name, k_raw, positions_to_show)


## 13. Load the analyzer module

This imports `analyze_qk_qwen3.py` as a module so notebook cells can call helper functions directly when useful. The CLI path is still used for the main smoke-test run below.


In [ ]:
import importlib.util
from pathlib import Path

ANALYZER_PATH = REPO_DIR / 'src' / 'dataset_generation' / 'qk_hook_attention' / 'analyze_qk_qwen3.py'
assert ANALYZER_PATH.exists(), f'Missing analyzer script at {ANALYZER_PATH}'

spec = importlib.util.spec_from_file_location('analyze_qk_qwen3', ANALYZER_PATH)
analyze_qk_qwen3 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(analyze_qk_qwen3)
print('Loaded analyzer module from:', ANALYZER_PATH)
print('Reusable helpers include:', ', '.join(name for name in ['reconstruct_single_head_qk', 'summarize_selected_rows', 'compute_query_block_stats', 'compute_full_attention_matrix'] if hasattr(analyze_qk_qwen3, name)))


## 14. Run a minimal layer/head analysis

This writes an analysis spec with first, middle, quarter, and last-token query positions plus simple named spans. It then runs `analyze_qk_qwen3.py` for layer 0/head 0, saves JSON output under `/content/qk_analysis_test`, saves selected full rows, and materializes the full `[T, T]` reconstructed attention matrix for the selected layer/head. The full matrix is intended only for this short-context smoke test.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

ANALYSIS_DIR = Path('/content/qk_analysis_test')
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

ANALYSIS_LAYER = 0
ANALYSIS_HEAD = 0
QUERY_POSITIONS = sorted({0, T // 4, T // 2, T - 1})
ANALYSIS_SPEC = {
    'query_positions': {
        'first': [0],
        'middle': [T // 2],
        'last': [-1],
        'manual_grid': QUERY_POSITIONS,
    },
    'spans': {
        'bos': [[0, 1]],
        'first_16': [[0, min(16, T)]],
        'middle_16': [[T // 2, min(T, T // 2 + 16)]],
        'last_16': [[max(0, T - 16), T]],
    },
    'local_windows': [16, 64, 256],
}

spec_path = ANALYSIS_DIR / 'analysis_spec_minimal.json'
out_json = ANALYSIS_DIR / f'stats_layer_{ANALYSIS_LAYER:02d}_head_{ANALYSIS_HEAD:02d}.json'
spec_path.write_text(json.dumps(ANALYSIS_SPEC, indent=2), encoding='utf-8')

cmd = [
    sys.executable,
    str(ANALYZER_PATH),
    '--cache-dir', str(CACHE_DIR),
    '--layer', str(ANALYSIS_LAYER),
    '--head', str(ANALYSIS_HEAD),
    '--spec-json', str(spec_path),
    '--out-json', str(out_json),
    '--device', 'cuda' if torch.cuda.is_available() else 'cpu',
    '--compute-dtype', 'fp32',
    '--key-block-size', '8192',
    '--query-block-size', '64',
    '--topk', '32',
    '--save-full-rows',
    '--save-full-matrix',
]
print('Running analyzer command:')
print(' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print('stdout:\n', result.stdout)
if result.stderr:
    print('stderr:\n', result.stderr)
result.check_returncode()
print('Analysis JSON:', out_json)


## 15. Display top-k, entropy, and span-mass results

This cell loads the analyzer JSON and displays compact pandas tables. If saved token strings are present, token text is included beside token positions.


In [ ]:
import json
import pandas as pd

analysis = json.loads(out_json.read_text(encoding='utf-8'))
rows = analysis['selected_query_rows']['rows']

topk_records = []
entropy_records = []
span_records = []
local_records = []
for qpos_str, item in rows.items():
    qpos = int(qpos_str)
    qtoken = item.get('token', '')
    labels = ','.join(item.get('labels', []))
    entropy_records.append({
        'layer': analysis['layer'],
        'head': analysis['head'],
        'query_position': qpos,
        'query_token': qtoken,
        'labels': labels,
        'entropy_nats': item.get('entropy_nats'),
    })
    for name, mass in item.get('span_mass', {}).items():
        span_records.append({
            'layer': analysis['layer'],
            'head': analysis['head'],
            'query_position': qpos,
            'span_name': name,
            'attention_mass': mass,
        })
    for window, mass in item.get('local_window_mass', {}).items():
        local_records.append({
            'layer': analysis['layer'],
            'head': analysis['head'],
            'query_position': qpos,
            'local_window': window,
            'attention_mass': mass,
        })
    for rank, tok in enumerate(item.get('topk', []), start=1):
        topk_records.append({
            'layer': analysis['layer'],
            'head': analysis['head'],
            'query_position': qpos,
            'rank': rank,
            'key_position': tok.get('position'),
            'key_token_id': tok.get('token_id'),
            'key_token': tok.get('token', ''),
            'prob': tok.get('prob'),
            'logit': tok.get('logit'),
        })

topk_df = pd.DataFrame(topk_records)
entropy_df = pd.DataFrame(entropy_records).sort_values('query_position')
span_df = pd.DataFrame(span_records).sort_values(['query_position', 'span_name'])
local_df = pd.DataFrame(local_records).sort_values(['query_position', 'local_window'])

print('Entropy by query position')
display(entropy_df)
print('Span masses')
display(span_df)
print('Local-window masses')
display(local_df)
print('Top-k attended positions')
display(topk_df.head(80))


## 16. Plot entropy and last-token span masses

Uses matplotlib only (no seaborn). The first plot shows query position versus entropy; the second shows named-span attention mass for the final token.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(entropy_df['query_position'], entropy_df['entropy_nats'], marker='o')
ax.set_xlabel('Query position')
ax.set_ylabel('Entropy (nats)')
ax.set_title(f'Layer {analysis["layer"]}, head {analysis["head"]}: entropy by query position')
ax.grid(True, alpha=0.3)
plt.show()

last_q = int(entropy_df['query_position'].max())
last_span = span_df[span_df['query_position'] == last_q]
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(last_span['span_name'], last_span['attention_mass'])
ax.set_xlabel('Span')
ax.set_ylabel('Attention mass')
ax.set_title(f'Layer {analysis["layer"]}, head {analysis["head"]}: span mass for query {last_q}')
ax.set_ylim(0, max(1.0, float(last_span['attention_mass'].max()) * 1.1 if not last_span.empty else 1.0))
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()


## 17. Critical-token inspection

Edit `SPECIAL_POSITIONS`, `NEEDLE_SPANS`, and `CRITICAL_QUERY_POSITIONS` to focus on particular tokens or ranges, then rerun the cell. It writes a separate JSON report and prints a compact span-mass table.


In [ ]:
SPECIAL_POSITIONS = []  # Example: [10, 128, T - 1]
NEEDLE_SPANS = {
    # 'needle_1': [(100, 120)],
}
CRITICAL_QUERY_POSITIONS = sorted({p for p in [0, T // 2, T - 1, *SPECIAL_POSITIONS] if 0 <= int(p) < T})

critical_spans = {
    'bos': [[0, 1]],
    'first_16': [[0, min(16, T)]],
    'last_16': [[max(0, T - 16), T]],
}
for name, intervals in NEEDLE_SPANS.items():
    critical_spans[name] = [[int(s), int(e)] for s, e in intervals]

critical_spec = {
    'query_positions': {
        'critical': CRITICAL_QUERY_POSITIONS,
        'last': [-1],
    },
    'spans': critical_spans,
    'local_windows': [16, 64, 256],
}
critical_spec_path = ANALYSIS_DIR / 'analysis_spec_critical.json'
critical_out_json = ANALYSIS_DIR / f'critical_layer_{ANALYSIS_LAYER:02d}_head_{ANALYSIS_HEAD:02d}.json'
critical_spec_path.write_text(json.dumps(critical_spec, indent=2), encoding='utf-8')

cmd = [
    sys.executable,
    str(ANALYZER_PATH),
    '--cache-dir', str(CACHE_DIR),
    '--layer', str(ANALYSIS_LAYER),
    '--head', str(ANALYSIS_HEAD),
    '--spec-json', str(critical_spec_path),
    '--out-json', str(critical_out_json),
    '--device', 'cuda' if torch.cuda.is_available() else 'cpu',
    '--compute-dtype', 'fp32',
    '--key-block-size', '8192',
    '--query-block-size', '64',
    '--topk', '16',
]
print('Running analyzer command:')
print(' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print('stdout:\n', result.stdout)
if result.stderr:
    print('stderr:\n', result.stderr)
result.check_returncode()

critical = json.loads(critical_out_json.read_text(encoding='utf-8'))
records = []
for qpos_str, item in critical['selected_query_rows']['rows'].items():
    for span_name, mass in item.get('span_mass', {}).items():
        records.append({
            'layer': critical['layer'],
            'head': critical['head'],
            'query_position': int(qpos_str),
            'span_name': span_name,
            'attention_mass': mass,
        })
critical_span_df = pd.DataFrame(records).sort_values(['query_position', 'span_name'])
display(critical_span_df)


## 18. Sanity checks

These checks validate basic attention invariants for saved full rows and the saved full matrix when available: row probabilities sum to one, causal masking removes future positions, span masses are probabilities, and entropy is in the expected range.


In [ ]:
import math
import warnings

warnings_list = []
row_path = analysis.get('full_rows_path')
if row_path and Path(row_path).exists():
    row_obj = torch.load(row_path, map_location='cpu')
    row_positions = row_obj['query_positions']
    row_probs = row_obj['rows'].float()
    row_sums = row_probs.sum(dim=-1)
    max_sum_error = float((row_sums - 1.0).abs().max().item()) if row_sums.numel() else 0.0
    print('Max |selected_row_sum - 1|:', max_sum_error)
    if max_sum_error > 1e-3:
        warnings_list.append(f'Selected probability row sums deviate from 1 by up to {max_sum_error:.3g}.')

    for i, qpos in enumerate(row_positions):
        future_mass = float(row_probs[i, int(qpos) + 1:].sum().item())
        if future_mass > 1e-5:
            warnings_list.append(f'Future-position mass for selected query {qpos} is {future_mass:.3g}.')
    print('Selected-row causal future-mask check complete for', len(row_positions), 'rows.')
else:
    warnings_list.append('No full attention rows found; rerun the analyzer cell with --save-full-rows to check selected row sums and causal masking.')

matrix_path = analysis.get('full_matrix_path')
if matrix_path and Path(matrix_path).exists():
    matrix_obj = torch.load(matrix_path, map_location='cpu')
    full_matrix = matrix_obj['matrix'].float()
    matrix_row_sums = full_matrix.sum(dim=-1)
    max_matrix_sum_error = float((matrix_row_sums - 1.0).abs().max().item()) if matrix_row_sums.numel() else 0.0
    upper_triangle_mass = float(torch.triu(full_matrix, diagonal=1).sum().item())
    print('Full matrix shape:', tuple(full_matrix.shape))
    print('Max |full_matrix_row_sum - 1|:', max_matrix_sum_error)
    print('Full matrix future-position mass:', upper_triangle_mass)
    if max_matrix_sum_error > 1e-3:
        warnings_list.append(f'Full matrix probability row sums deviate from 1 by up to {max_matrix_sum_error:.3g}.')
    if upper_triangle_mass > 1e-5:
        warnings_list.append(f'Full matrix has {upper_triangle_mass:.3g} total future-position mass.')

    if row_path and Path(row_path).exists():
        selected_from_matrix = full_matrix[torch.tensor(row_positions, dtype=torch.long)]
        row_matrix_diff = float((selected_from_matrix - row_probs).abs().max().item()) if row_probs.numel() else 0.0
        print('Max |saved selected rows - full matrix rows|:', row_matrix_diff)
        if row_matrix_diff > 1e-5:
            warnings_list.append(f'Saved selected rows differ from the saved full matrix by up to {row_matrix_diff:.3g}.')
else:
    warnings_list.append('No full attention matrix found; rerun the analyzer cell with --save-full-matrix to check full-matrix invariants.')

bad_span = span_df[(span_df['attention_mass'] < -1e-6) | (span_df['attention_mass'] > 1 + 1e-6)]
if not bad_span.empty:
    warnings_list.append(f'{len(bad_span)} span masses fall outside [0, 1].')
else:
    print('Span masses are within [0, 1].')

for _, row in entropy_df.iterrows():
    qpos = int(row['query_position'])
    entropy = float(row['entropy_nats'])
    max_entropy = math.log(qpos + 1) if qpos >= 0 else 0.0
    if entropy < -1e-6 or entropy > max_entropy + 1e-3:
        warnings_list.append(f'Entropy {entropy:.6g} for query {qpos} is outside [0, log({qpos + 1})={max_entropy:.6g}].')
print('Entropy bounds check complete.')

if warnings_list:
    print('\nWARNINGS:')
    for msg in warnings_list:
        warnings.warn(msg)
else:
    print('All sanity checks passed.')


## 19. Optional short-context validation against Hugging Face attentions

This is **disabled by default** because `output_attentions=True` materializes full attention matrices and should only be used for very short prompts. If you want to compare the reconstructed full `[T, T]` matrix from the Q/K cache against Hugging Face, set `RUN_OPTIONAL_HF_VALIDATION = True`. The cell loads the same model with `attn_implementation='eager'`, runs the cached prompt with `output_attentions=True`, and compares the selected layer/head matrix against `analysis['full_matrix_path']`.


In [ ]:
RUN_OPTIONAL_HF_VALIDATION = False
VALIDATION_ATOL = 3e-2
VALIDATION_RTOL = 3e-2
VALIDATION_MODEL_DTYPE = torch.bfloat16

if RUN_OPTIONAL_HF_VALIDATION:
    from transformers import AutoModelForCausalLM

    matrix_path = analysis.get('full_matrix_path')
    assert matrix_path and Path(matrix_path).exists(), 'Run the analyzer cell with --save-full-matrix before HF validation.'

    reconstructed = torch.load(matrix_path, map_location='cpu')['matrix'].float()
    input_ids = torch.load(CACHE_DIR / 'input_ids.pt', map_location='cpu')
    attention_mask = torch.load(CACHE_DIR / 'attention_mask.pt', map_location='cpu')
    position_ids = torch.load(CACHE_DIR / 'position_ids.pt', map_location='cpu')

    print('Loading model for Hugging Face attention validation with attn_implementation="eager"...')
    hf_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=VALIDATION_MODEL_DTYPE,
        device_map='auto',
        attn_implementation='eager',
    )
    hf_model.eval()

    input_device = hf_model.get_input_embeddings().weight.device
    model_inputs = {
        'input_ids': input_ids.to(input_device),
        'attention_mask': attention_mask.to(input_device),
        'position_ids': position_ids.to(input_device),
    }

    with torch.inference_mode():
        hf_outputs = hf_model(
            **model_inputs,
            use_cache=False,
            output_attentions=True,
            return_dict=True,
            logits_to_keep=1,
        )

    hf_matrix = hf_outputs.attentions[ANALYSIS_LAYER][0, ANALYSIS_HEAD].detach().cpu().float()
    diff = (hf_matrix - reconstructed).abs()
    max_abs_diff = float(diff.max().item())
    mean_abs_diff = float(diff.mean().item())
    row_sum_error = float((reconstructed.sum(dim=-1) - 1.0).abs().max().item())
    hf_row_sum_error = float((hf_matrix.sum(dim=-1) - 1.0).abs().max().item())
    future_mass = float(torch.triu(reconstructed, diagonal=1).sum().item())
    hf_future_mass = float(torch.triu(hf_matrix, diagonal=1).sum().item())

    print('Reconstructed matrix shape:', tuple(reconstructed.shape))
    print('HF matrix shape:', tuple(hf_matrix.shape))
    print('Max absolute difference:', max_abs_diff)
    print('Mean absolute difference:', mean_abs_diff)
    print('Reconstructed max row-sum error:', row_sum_error)
    print('HF max row-sum error:', hf_row_sum_error)
    print('Reconstructed future-position mass:', future_mass)
    print('HF future-position mass:', hf_future_mass)

    assert reconstructed.shape == hf_matrix.shape
    assert torch.allclose(reconstructed, hf_matrix, atol=VALIDATION_ATOL, rtol=VALIDATION_RTOL), (
        f'Reconstructed attention differs from HF output_attentions=True: '
        f'max_abs_diff={max_abs_diff:.6g}, mean_abs_diff={mean_abs_diff:.6g}'
    )
    print('Full attention matrix validation passed.')

    del hf_outputs, hf_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print('Skipping optional Hugging Face output_attentions=True validation. Set RUN_OPTIONAL_HF_VALIDATION = True to run it.')
